### Data Quality Checks

Purpose:
- Validate dbt-generated models
- Detect data quality issues
- Ensure analytical tables are reliable

dbt Models Tested:
- models/staging/stg_customer_support_tickets.sql
- models/marts/dim_customers.sql
- models/marts/dim_products.sql
- models/marts/int_ticket_performance.sql
- models/marts/fact_ticket_metrics.sql

Import libraries

In [17]:
import duckdb
import pandas as pd
import numpy as np

from datetime import datetime

Connect to DuckDB

In [18]:
conn = duckdb.connect(
    "../customer_support.duckdb",
    read_only=True
)

print("DuckDB connection successful")

DuckDB connection successful


Create Query Helper Function

In [19]:
def run_query(query):
    return conn.execute(query).df()

Show available tables

In [20]:
run_query("""
SHOW TABLES;
""")

,name
0,dim_customers
1,dim_products
2,fact_ticket_metrics
3,int_ticket_performance
4,stg_customer_support_tickets


Row counts check

In [21]:
run_query("""
SELECT
    'stg_customer_support_tickets' AS table_name,
    COUNT(*) AS records
FROM main.stg_customer_support_tickets

UNION ALL

SELECT
    'dim_customers',
    COUNT(*)
FROM main.dim_customers

UNION ALL

SELECT
    'dim_products',
    COUNT(*)
FROM main.dim_products

UNION ALL

SELECT
    'fact_ticket_metrics',
    COUNT(*)
FROM main.fact_ticket_metrics;
""")

,table_name,records
0,stg_customer_support_tickets,8469
1,dim_customers,8469
2,dim_products,42
3,fact_ticket_metrics,8791


Null values check

In [22]:
run_query("""
SELECT
    COUNT(*) AS total_records,

    SUM(
        CASE WHEN customer_id IS NULL 
        THEN 1 ELSE 0 END
    ) AS missing_customer_id,

    SUM(
        CASE WHEN customer_email IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_email

FROM main.dim_customers;
""")

,total_records,missing_customer_id,missing_email
0,8469,0.0,0.0


Duplicate checks for Customers

In [23]:
run_query("""
SELECT
    customer_id,
    COUNT(*) AS duplicate_count

FROM main.dim_customers

GROUP BY customer_id

HAVING COUNT(*) > 1;
""")

,customer_id,duplicate_count


Duplicate checks for Tickets

In [24]:
run_query("""
SELECT
    ticket_id,
    COUNT(*) AS duplicate_count

FROM main.stg_customer_support_tickets

GROUP BY ticket_id

HAVING COUNT(*) > 1;
""")

,ticket_id,duplicate_count


Referential Integrity

Check that fact tables correctly link to dimensions.

In [25]:
run_query("""
SELECT COUNT(*) AS unmatched_customers

FROM main.fact_ticket_metrics f

LEFT JOIN main.dim_customers c
ON f.customer_id = c.customer_id

WHERE c.customer_id IS NULL;
""")

,unmatched_customers
0,0


Check customer age validity:

In [26]:
run_query("""
SELECT
    MIN(customer_age) AS minimum_age,
    MAX(customer_age) AS maximum_age

FROM main.dim_customers;
""")

,minimum_age,maximum_age
0,18,70


Customer satisfaction must be between 1 and 5

In [27]:
run_query("""
SELECT DISTINCT
    customer_satisfaction_rating

FROM main.fact_ticket_metrics

ORDER BY 1;
""")

,customer_satisfaction_rating
0,1.0
1,2.0
2,3.0
3,4.0
4,5.0
5,NaN


Product Validation

In [28]:
run_query("""
SELECT
    COUNT(*) AS products,
    COUNT(DISTINCT product_name) AS unique_products

FROM main.dim_products;
""")

,products,unique_products
0,42,42


Response time correction

In [29]:
run_query("""
SELECT
    response_time_quality_flag,
    COUNT(*) AS records
FROM main.fact_ticket_metrics
GROUP BY response_time_quality_flag;
""")

,response_time_quality_flag,records
0,Corrected,1426
1,Valid,7365


Validate resolution hours calculation

In [30]:
run_query("""
SELECT
    MIN(resolution_hours) AS min_resolution,
    ROUND(AVG(resolution_hours),2) AS avg_resolution,
    MAX(resolution_hours) AS max_resolution
FROM main.fact_ticket_metrics;
""")

,min_resolution,avg_resolution,max_resolution
0,0.0,7.73,23.47


In [31]:
run_query("""
SELECT
    ticket_id,
    COUNT(*) AS occurrences

FROM main.fact_ticket_metrics

GROUP BY ticket_id

HAVING COUNT(*) > 1

ORDER BY occurrences DESC;
""")

,ticket_id,occurrences
0,6985,4
1,7346,4
2,7275,4
3,2217,4
4,3008,4
...,...,...
283,7802,2
284,7936,2
285,7998,2
286,8108,2


Data Quality Summary

In [32]:
quality_summary = pd.DataFrame({
    "Check": [
        "Model record counts",
        "Customer uniqueness",
        "Ticket uniqueness",
        "Missing customer IDs",
        "Fact dimension relationships"
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ]
})

quality_summary

,Check,Status
0,Model record counts,PASS
1,Customer uniqueness,PASS
2,Ticket uniqueness,PASS
3,Missing customer IDs,PASS
4,Fact dimension relationships,PASS


Close connection

In [33]:
conn.close()

print("Connection closed")

Connection closed
